<a href="https://colab.research.google.com/github/MengOonLee/LLM/blob/main/References/LangChain/LangChainAcademy/LangChain/Foundation/CreateAgent/ipynb/1.2_web_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%bash
pip install --no-cache-dir -qU \
    langchain langgraph langchain-core langchain-community \
    langchain-openai tavily-python ollama

In [1]:
import warnings
warnings.filterwarnings("ignore")
import dotenv

_ = dotenv.load_dotenv(dotenv_path=".env", override=True)

In [2]:
import os
from langchain import chat_models

llm = chat_models.init_chat_model(
    model="qwen3.5:cloud",
    model_provider="openai",
    base_url="https://ollama.com/v1",
    api_key=os.environ["OLLAMA_API_KEY"],
    temperature=0
)

## Without web search

In [3]:
import time
from langchain import agents, messages

agent = agents.create_agent(model=llm)

start_time = time.time()

messages = [messages.HumanMessage(content="""
How up to date is your training knowledge?
""")]
response = agent.invoke(input={"messages": messages})
for m in response["messages"]:
    m.pretty_print()

end_time = time.time() - start_time
print(f"Time taken: %.2fs seconds"%(end_time))

================================ Human Message =================================


How up to date is your training knowledge?

================================== Ai Message ==================================

My training data is current up to **2026**. This means my knowledge and capabilities are based on information available up to that point. For events, developments, or data after this date, I may not have accurate or updated information. Always verify time-sensitive details through reliable sources! 😊
Time taken: 2.35s seconds


## Add web search tool

### Tavily Web Search

In [42]:
import tavily
import typing
from langchain import tools

tavily_client = tavily.TavilyClient()

@tools.tool
def web_search(query: str) -> typing.Dict[str, typing.Any]:
    """
    Search the web for information
    """
    return tavily_client.search(query=query)

web_search.invoke(input="""
List of national public holidays in Malaysia for 2026.
""")

{'query': 'List of national public holidays in Malaysia for 2026.',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://calendarific.com/holidays/2026/MY',
   'title': 'Malaysia Public Holidays in 2026 - Calendarific',
   'content': "| **Federal Territory Day** | Sunday, February 1, 2026 |. | **Federal Territory Day Observed** | Monday, February 2, 2026 |. | **Day off for Thaipusam** | Monday, February 2, 2026 |. | **Valentine's Day** | Saturday, February 14, 2026 |. | **Chinese New Year's Day** | Tuesday, February 17, 2026 |. | **Second Day of Chinese New Year** | Wednesday, February 18, 2026 |. | **Independence Day Declaration Day** | Friday, February 20, 2026 |. | **Second Day of Harvest Festival Observed** | Monday, June 1, 2026 |. | **Sultan of Kedah's Birthday** | Sunday, June 21, 2026 |. | **Birthday of the Sultan of Pahang** | Friday, July 31, 2026 |. | **Malaysia Day** | Wednesday, September 16, 2026 |. | **Holiday for Birthday of the Su

In [43]:
import pydantic
from langchain import agents
import typing

class HolidayInfo(pydantic.BaseModel):
    country_code: str
    date: str
    name: str

class HolidayList(pydantic.BaseModel):
    holidays: typing.List[HolidayInfo]

system_prompt = """
You are a strict data extraction assistant.

Rules:
- Always use the web_search tool
- Return ONLY valid JSON matching the schema
- Do not include explanations

Schema:
HolidayList with field:
- holidays: list of HolidayInfo
"""

agent = agents.create_agent(
    model=llm,
    tools=[web_search],
    system_prompt=system_prompt,
    response_format=HolidayList
)

In [44]:
import time
from langchain import messages
import json

start_time = time.time()

messages = [messages.HumanMessage(content="""
List of national public holidays in Malaysia for 2026.
""")]
response = agent.invoke(input={"messages": messages})

for m in response["messages"]:
    m.pretty_print()

holiday_list = response["structured_response"]
with open("malaysia_holidays_2026.jl", "w") as f:
    for h in holiday_list.holidays:
        f.write(json.dumps(h.model_dump()) + "\n")

end_time = time.time() - start_time
print(f"Time taken: %.2fs"%(end_time))

================================ Human Message =================================


List of national public holidays in Malaysia for 2026.

================================== Ai Message ==================================
Tool Calls:
  web_search (call_fwxg35hc)
 Call ID: call_fwxg35hc
  Args:
    query: Malaysia national public holidays 2026
================================= Tool Message =================================
Name: web_search

{"query": "Malaysia national public holidays 2026", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://calendarific.com/holidays/2026/MY", "title": "Malaysia Public Holidays in 2026 - Calendarific", "content": "| **Federal Territory Day** | Sunday, February 1, 2026 |. | **Federal Territory Day Observed** | Monday, February 2, 2026 |. | **Day off for Thaipusam** | Monday, February 2, 2026 |. | **Valentine's Day** | Saturday, February 14, 2026 |. | **Chinese New Year's Day** | Tuesday, February 17, 2026 |. | **Second 

### Ollama Web Search

In [ ]:
import os
from ollama import Client
from typing import Dict, Any
from langchain.tools import tool

client = Client(
    host="https://ollama.com/api/web_search",
    headers={"Authorization": "Bearer " + os.environ["OLLAMA_API_KEY"]}
)

@tool
def web_search(query: str, domains: list) -> Dict[str, Any]:
    """
    Search the web for information
    """

    domains = " OR ".join([f"site:{d}" for d in domains])
    query = f"{query} ({domains})"
    return client.web_search(query=query)

web_search.invoke(input={
"query": "List of public holidays in Malaysia for the year 2026.",
"domains": ["timeanddate.com", "officeholidays.com"]
})

WebSearchResponse(results=[WebSearchResult(content="Holidays and Observances in Malaysia in 2026\n\nShowing: all Public holidaysPublic holidays and non-working daysHolidays and some observancesHolidays (incl. some local) and observancesHolidays (incl. all local) and observancesAll holidays/observances/religious eventsCustom – choose holidays... For: 200020012002200320042005200620072008200920102011201220132014201520162017201820192020202120222023202420252026Today20272028202920302031203220332034203520362037203820392040 Select AllClear AllReset to Default Federal/National Holidays (15)Common Local Holidays (14)Local holidays (28)Important observances (4)Seasons (4)Major Hinduism (0)\n\n## Holidays and Observances in Malaysia in 2026\n\n| Date | Name | Type | Details |\n| --- | --- | --- | --- |\n| Jan 1 | Thursday | New Year's Day | State Holiday | All except JHR, KDH, KTN, PLS, TRG |\n| Jan 14 | Wednesday | Birthday of Yang di-Pertuan Besar | State Holiday | NSN |\n| Jan 17 | Saturday | I

In [ ]:
from langchain.agents import create_agent

system_prompt = """
You are a helpful assistant.
When using the web_search tool, you MUST pass:
- query: string
- domains: list of domains

Please keep the below structure.
Country code: The country code of the holiday.
Date: The date of the holiday.
Name: The name of the holiday.
"""

agent = create_agent(
    model=llm,
    tools=[web_search],
    system_prompt=system_prompt
)

In [ ]:
import time
from langchain.messages import HumanMessage

start_time = time.time()

messages = [HumanMessage(content="""
List of public holidays in Malaysia for the year 2026.

Use the web_search tool with:
- query: "Malaysia public holidays 2026"
- domains: ["timeanddate.com", "officeholidays.com"]
""")]
response = agent.invoke(input={"messages": messages})

for m in response["messages"]:
    m.pretty_print()

end_time = time.time() - start_time
print(f"Time taken: %.2fs seconds"%(end_time))

================================ Human Message =================================


List of public holidays in Malaysia for the year 2026.

Use the web_search tool with:
- query: "Malaysia public holidays 2026"
- domains: ["timeanddate.com", "officeholidays.com"]

================================== Ai Message ==================================
Tool Calls:
  web_search (42f12e20-556f-4075-b0e4-735cb3602f07)
 Call ID: 42f12e20-556f-4075-b0e4-735cb3602f07
  Args:
    domains: ['timeanddate.com', 'officeholidays.com']
    query: Malaysia public holidays 2026
================================= Tool Message =================================
Name: web_search

results=[WebSearchResult(content="Holidays and Observances in Malaysia in 2026\n\nShowing: all Public holidaysPublic holidays and non-working daysHolidays and some observancesHolidays (incl. some local) and observancesHolidays (incl. all local) and observancesAll holidays/observances/religious eventsCustom – choose holidays... For: 2000200